# Get To Know A Dataset: Human Modification Forecasts 2025–2040

This notebook is a guided tour of the [Probabilistic Forecasts of Global Human Modification, 2025–2040](https://registry.opendata.aws/tnc-human-modification-forecasts) dataset from The Nature Conservancy. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

**What is in the dataset?** Human Modification (HM) is a continuous, unitless index from 0 (no detectable human modification) to 1 (complete modification). It combines 16 anthropogenic threats, mapped to the IUCN threat taxonomy, into one cumulative measure of human pressure on land ([Theobald et al. 2025](https://doi.org/10.1038/s41597-025-04892-2)). This dataset holds:

* **Forecasts** of HM for 2025, 2030, 2035 and 2040 at ~1 km resolution. These are full probability distributions rather than single values. Each land pixel has 64 quantiles of its forecast distribution, produced by a convolutional LSTM trained on the 1990–2020 HM record ([Moncrieff et al. 2026](https://doi.org/10.32942/X2S11G)).
* **Observed** HM for 1990–2020 (every 5 years) on exactly the same grid, so that you can work with one seamless 1990–2040 series.

Full documentation, including a data dictionary, is at [github.com/GMoncrieff/open-data-examples](https://github.com/GMoncrieff/open-data-examples/tree/main/human-modification-forecasts).

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The S3 bucket `tnc-human-modification` (region `us-east-1`) contains a single [Icechunk](https://icechunk.io) repository under the prefix `hm_probability_forecast/`. An Icechunk repository is a versioned, transactional store of [Zarr](https://zarr.dev) arrays: think of it as "git for multidimensional arrays".

The objects under that prefix are Icechunk's internal storage: snapshots, manifests, chunk files and transaction logs. Their names are content hashes and they are **not meant to be read one by one**. Instead, you open the repository with the `icechunk` Python library and see a normal Zarr hierarchy with two groups:

| Zarr group | Variable | Dimensions | Shape | What it holds |
|---|---|---|---|---|
| `hm_prob_forecast` | `hm` | `year, percentile, latitude, longitude` | 4 × 64 × 17,111 × 40,000 | Forecast quantile function for 2025, 2030, 2035, 2040 |
| `hm_observed` | `hm` | `year, latitude, longitude` | 7 × 17,111 × 40,000 | Observed HM for 1990, 1995, …, 2020 |

Both groups share the same grid: EPSG:4326 at 0.009° (~1 km at the equator), covering land from 84° N to 70° S, with oceans and permanent inland water set to missing. Data are read from the repository's `main` branch. Every change to an Icechunk repository creates a new snapshot, so you can pin an analysis to an exact version by recording the snapshot ID (or a tag, where one exists).

We'll start by looking at the raw objects with `boto3`, then open the repository the intended way.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# icechunk >= 1.0
# zarr >= 3.0
# xarray >= 2025.1
# numpy >= 2.0
# matplotlib >= 3.8
# boto3 >= 1.34

# Built-ins
from pprint import pprint

# Installed libraries
import boto3
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import zarr
from botocore import UNSIGNED
from botocore.config import Config

First we define the location of the dataset and list the top level of the bucket. The bucket is public, so requests are unsigned and no AWS account is needed.

In [ ]:
# Location of the dataset
BUCKET = "tnc-human-modification"
PREFIX = "hm_probability_forecast"
REGION = "us-east-1"

s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))

# Top-level prefixes in the bucket
for item in s3.list_objects_v2(Bucket=BUCKET, Delimiter="/").get("CommonPrefixes", []):
    print(item["Prefix"])

Inside `hm_probability_forecast/` are the Icechunk repository's own objects. Depending on the Icechunk format version you will see a small repository metadata object (`repo` or `config.yaml`/`refs/`) plus `snapshots/`, `manifests/`, `chunks/` and `transactions/`. The `chunks/` prefix holds almost all of the bytes.

In [ ]:
# Objects and prefixes directly inside the Icechunk repository
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}/", Delimiter="/")
for item in resp.get("CommonPrefixes", []):
    print("prefix:", item["Prefix"])
for item in resp.get("Contents", []):
    print("object:", item["Key"], f'({item["Size"]:,} bytes)')

Now we open the repository with `icechunk`. Opening is cheap: only small metadata objects are read. A *read-only session* on the `main` branch gives us a Zarr store, and we can walk its hierarchy.

In [ ]:
storage = icechunk.s3_storage(bucket=BUCKET, prefix=PREFIX, region=REGION, anonymous=True)
repo = icechunk.Repository.open(storage)

print("branches:", repo.list_branches())
print("tags:    ", repo.list_tags())

session = repo.readonly_session(branch="main")
root = zarr.open_group(session.store, mode="r")

# Walk the Zarr hierarchy
for group_name, group in root.groups():
    print(f"\n/{group_name}")
    for name, arr in group.arrays():
        print(f"    {name:<11} shape={arr.shape!s:<28} dtype={arr.dtype}")

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

**Zarr v3** is an open, community-standard format for chunked, compressed N-dimensional arrays. Each array is split into a regular grid of chunks that are compressed and stored as separate objects, so a program can read just the chunks it needs over HTTP rather than downloading a whole file. **Icechunk** is an open-source storage engine that sits underneath Zarr and adds transactional, versioned updates (branches, tags and snapshots) and a fast Rust I/O layer for object storage. To a user, an Icechunk repository looks like a Zarr store.

We chose this format because:

* **The forecast is a distribution, not a map.** Each pixel has 64 quantiles × 4 years. That is ~350 GB uncompressed (~70 GB compressed) for the forecast group, far too big to download for a regional question. With chunked storage you read only the pixels you need.
* **The quantile axis is never split.** Each chunk holds *all 64 percentiles* for one year over a 256 × 320 pixel window (~10 MB uncompressed). One read therefore returns the complete forecast distribution for every pixel in that window. Chunks are grouped into 1024 × 1280 pixel *shards* (one object per shard) to keep the object count manageable.
* **Versioning.** If the forecasts are revised (for example, when a new HM epoch is released), Icechunk snapshots and tags let you reproduce an analysis against the exact version you used.

**Dataset-specific encoding.** HM values are stored as unsigned 16-bit integers. Multiply by `scale_factor` = 1/65535 to recover HM on the 0–1 scale. The value 65535 is the fill value (ocean, inland water, Antarctica), so the largest representable HM is 65534/65535 ≈ 0.99998. These CF attributes are on the arrays, and **xarray applies them for you**: you get floating-point HM with `NaN` for missing pixels.

**The `percentile` coordinate** gives the 64 quantile levels *in percent*, from 0.0248 to 99.99. The 2.5th, 50th and 97.5th percentiles are exact levels, and the rest are spaced more densely in the tails. The model represents each pixel's distribution as a monotone, piecewise-linear quantile function through these 64 points, so linear interpolation between levels is consistent with how the forecasts were produced.

**Recommended tools.** Use `xarray` + `icechunk` (shown below) for analysis, adding `dask` to scale up. GIS users who just want the mean, the 2.5th and 97.5th percentiles, and the probability of exceeding HM thresholds as GeoTIFFs can download them from [Zenodo](https://doi.org/10.5281/zenodo.19700998). An interactive browser is on the [Earthmover Data Marketplace](https://app.earthmover.io/marketplace/6abb60fba504351ef2a29c6f).

**AWS services.** Reads are fastest from compute in the same region (`us-east-1`), e.g. [Amazon EC2](https://aws.amazon.com/ec2/) or [Amazon SageMaker AI](https://aws.amazon.com/sagemaker-ai/) notebooks. For continental or global analyses, a Dask cluster on EC2 (e.g. via [Coiled](https://coiled.io) or Dask Gateway) can process the whole forecast in parallel, one chunk per task.

### Q: Can you show us an example of downloading and loading data from your dataset?

We open each group as a lazy `xarray.Dataset`. Nothing beyond metadata is downloaded until we ask for values. We pass `chunks=None` so xarray reads lazily without dask. For larger jobs, `chunks={}` gives dask arrays aligned to the stored chunks.

In [ ]:
obs = xr.open_zarr(session.store, group="hm_observed", consolidated=False, chunks=None)
fc = xr.open_zarr(session.store, group="hm_prob_forecast", consolidated=False, chunks=None)

obs

In [ ]:
fc

In [ ]:
print(f"Forecast group: {fc.hm.size * 2 / 1e9:,.0f} GB uncompressed as uint16 (we will only ever read small pieces)")
print("Stored encoding:", {k: fc.hm.encoding.get(k) for k in ("dtype", "scale_factor", "_FillValue", "chunks", "shards")})
print("\nForecast years:", fc.year.values)
print("Observed years:", obs.year.values)
print(f"\n{fc.percentile.size} percentile levels (percent):")
print(", ".join(f"{p:.4g}" for p in fc.percentile.values))

Let's pull out the complete forecast distribution for a single 1 km pixel. We choose a point in the farmland–rangeland mosaic of central Kenya, one of the regions highlighted in the accompanying paper. `.sel(..., method="nearest")` snaps to the closest pixel. Reading one pixel touches a single chunk per year.

In [ ]:
# A pixel in central Kenya (latitude, longitude in degrees)
point = dict(latitude=0.05, longitude=37.05)

px_fc = fc.hm.sel(point, method="nearest").load()    # (year, percentile)
px_obs = obs.hm.sel(point, method="nearest").load()  # (year,)

print("Observed HM:")
print(px_obs.to_series().round(3).to_string())

# 2.5th, 50th and 97.5th percentiles are exact levels in the stored grid
summary = px_fc.sel(percentile=[2.5, 50.0, 97.5]).to_pandas().round(3)
summary.columns = ["2.5th pct", "median", "97.5th pct"]
print("\nForecast HM:")
print(summary.to_string())

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

**1. The forecast as a fan chart.** Because observed and forecast HM share one grid, a pixel's 1990–2020 history and its 2025–2040 forecast distribution join into one series. The shaded bands are central prediction intervals built from the stored percentiles.

In [ ]:
pct = fc.percentile.values

def nearest_level(p):
    # The stored percentile closest to p (2.5, 50 and 97.5 are exact)
    return pct[np.abs(pct - p).argmin()]

years_obs, years_fc = px_obs.year.values, px_fc.year.values
# Anchor the forecast bands at the last observation so the fan starts from 2020
anchor_year, anchor_val = years_obs[-1], float(px_obs.values[-1])

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=110)
for lo, hi, alpha, label in [(2.5, 97.5, 0.25, "95% interval"), (25, 75, 0.45, "50% interval")]:
    q_lo = px_fc.sel(percentile=nearest_level(lo)).values
    q_hi = px_fc.sel(percentile=nearest_level(hi)).values
    ax.fill_between(np.r_[anchor_year, years_fc], np.r_[anchor_val, q_lo], np.r_[anchor_val, q_hi],
                    color="#b73779", alpha=alpha, lw=0, label=label)
ax.plot(np.r_[anchor_year, years_fc], np.r_[anchor_val, px_fc.sel(percentile=50.0).values],
        color="#641a80", lw=2, ls="--", label="forecast median")
ax.plot(years_obs, px_obs.values, color="black", lw=2, marker="o", ms=4, label="observed")
ax.axhline(0.1, color="grey", lw=0.8, ls=":")
ax.text(1990.5, 0.1, " natural-land threshold (HM = 0.1)", va="bottom", fontsize=8, color="grey")
ax.set(xlabel="Year", ylabel="Human Modification (0–1)",
       title=f"HM at {point['latitude']}°, {point['longitude']}°: observed 1990–2020, forecast 2025–2040")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

**2. Regional maps from the full distribution.** Next we read a 2.5° × 2.5° window over central Kenya (~280 × 280 km) for all 64 percentiles of the 2040 forecast. That is a handful of chunks, a few hundred MB at most.

From the quantile function we derive two quantities that are not stored directly but are often what decisions need:

* the **mean** (expected HM), approximated by integrating the quantile function over its levels;
* the **probability that HM exceeds a threshold**, found by inverting the piecewise-linear quantile function. Here we use HM > 0.1, the boundary of "natural" land used in the paper.

We write these as two small helpers that work on any array with a `percentile` dimension, so you can reuse them.

In [ ]:
U = fc.percentile.values / 100.0  # quantile levels as probabilities

def _exceedance(q, u, threshold):
    # P(HM > threshold) for quantile values q (..., P) at levels u (P,)
    k = (q <= threshold).sum(axis=-1)                    # number of levels at or below threshold
    hi = np.clip(k, 1, len(u) - 1)
    lo = hi - 1
    q_lo = np.take_along_axis(q, lo[..., None], -1)[..., 0]
    q_hi = np.take_along_axis(q, hi[..., None], -1)[..., 0]
    frac = np.where(q_hi > q_lo, (threshold - q_lo) / np.where(q_hi > q_lo, q_hi - q_lo, 1), 1.0)
    cdf = u[lo] + (u[hi] - u[lo]) * np.clip(frac, 0, 1)
    cdf = np.where(k == 0, u[0], np.where(k == len(u), u[-1], cdf))
    out = 1.0 - cdf
    return np.where(np.isnan(q).any(axis=-1), np.nan, out)

def prob_exceed(da, threshold):
    # Probability that HM exceeds `threshold`, from a DataArray with a 'percentile' dimension
    return xr.apply_ufunc(_exceedance, da, kwargs=dict(u=U, threshold=threshold),
                          input_core_dims=[["percentile"]])

def quantile_mean(da):
    # Expected HM: trapezoidal integral of the quantile function over u, extended flat to u=0 and u=1
    u_ext = np.r_[0.0, U, 1.0]
    q = da.transpose(..., "percentile").values
    q_ext = np.concatenate([q[..., :1], q, q[..., -1:]], axis=-1)
    return da.isel(percentile=0, drop=True).copy(data=np.trapezoid(q_ext, u_ext, axis=-1))

In [ ]:
# Central Kenya window. Latitude is stored north-to-south, so slice from north to south.
window = dict(latitude=slice(1.25, -1.25), longitude=slice(35.75, 38.25))

obs_2020 = obs.hm.sel(year=2020, **window).load()
fc_2040 = fc.hm.sel(year=2040, **window).load()    # (percentile, latitude, longitude)
print(f"Read {fc_2040.size:,} values  ->  {fc_2040.sizes['latitude']} x {fc_2040.sizes['longitude']} pixels x 64 percentiles")

mean_2040 = quantile_mean(fc_2040)
p_gt01_2040 = prob_exceed(fc_2040, 0.1)
width_2040 = fc_2040.sel(percentile=97.5) - fc_2040.sel(percentile=2.5)

In [ ]:
panels = [
    (obs_2020, "Observed HM, 2020", "magma", (0, 1)),
    (mean_2040, "Forecast mean HM, 2040", "magma", (0, 1)),
    (mean_2040 - obs_2020, "Expected change, 2020→2040", "PuOr_r", (-0.15, 0.15)),
    (width_2040, "Width of 95% interval, 2040", "viridis", (0, None)),
]
fig, axes = plt.subplots(2, 2, figsize=(11, 10), dpi=100, constrained_layout=True)
for ax, (da, title, cmap, (vmin, vmax)) in zip(axes.flat, panels):
    im = da.plot.imshow(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=False, add_labels=False)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title)
    ax.set_aspect("equal")
fig.suptitle("Central Kenya: present, expected future, and uncertainty", fontsize=14)
plt.show()

**3. Where is natural land at risk?** The mean surface highlights places where large change is *expected*, which is mostly next to existing development. The exceedance probability tells a different story. Among pixels that are natural today (HM ≤ 0.1 in 2020), many carry a small but non-zero chance of crossing the threshold by 2040, even far from current frontiers. This is the information a precautionary screen needs, and it is lost if you look only at the mean.

In [ ]:
natural_2020 = obs_2020 <= 0.1

fig, ax = plt.subplots(figsize=(7.5, 6.5), dpi=110)
obs_2020.where(~natural_2020).plot.imshow(ax=ax, cmap="Greys", vmin=-1, vmax=1.5, add_colorbar=False, add_labels=False)
im = p_gt01_2040.where(natural_2020).plot.imshow(ax=ax, cmap="inferno", vmin=0, vmax=0.5,
                                                 add_colorbar=False, add_labels=False)
fig.colorbar(im, ax=ax, shrink=0.8, label="P(HM > 0.1 in 2040)")
ax.set_title("Probability that land natural in 2020 loses natural status by 2040\n(grey: already non-natural in 2020)")
ax.set_aspect("equal")
plt.show()

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**How much natural land is likely to lose its natural status by 2040, and how should we report that from per-pixel forecasts?**

In the paper we estimate that about 4.5 million km² of land that was natural in 2020 (HM ≤ 0.1) could lose that status by 2040 globally. Here is the same calculation for the central Kenya window, with one important lesson about aggregating probabilistic forecasts.

The forecast distributions are generated **independently for each pixel**. They do not describe how pixels co-vary. Two consequences follow:

* ✅ **Expectations add up.** The expected area lost in a region is the sum, over pixels, of pixel area × P(HM > 0.1). This is valid whatever the correlation between pixels, because the expectation of a sum is the sum of the expectations. Two 1 km² pixels each with a 0.2 chance of crossing the threshold contribute 0.4 km² to the expected loss.
* ✅ **Proportions at a percentile are fine.** "X km² of natural land has a 97.5th percentile above 0.1" is a valid statement about how widespread the plausible risk is.
* ❌ **Percentiles do not add up.** Summing the area that crosses the threshold in the 97.5th percentile map is *not* the regional 97.5th percentile of loss. That would require every pixel to realise its unlikely upper outcome at the same time.

First, the area of each pixel. Pixels are 0.009° squares, so their area shrinks with latitude.

In [ ]:
R_EARTH_KM = 6371.0088
res = 0.009  # degrees

def pixel_area_km2(lat):
    # Area of a 0.009-degree cell centred on each latitude (spherical Earth)
    lat_n, lat_s = np.deg2rad(lat + res / 2), np.deg2rad(lat - res / 2)
    return R_EARTH_KM**2 * np.deg2rad(res) * (np.sin(lat_n) - np.sin(lat_s))

area = pixel_area_km2(obs_2020.latitude) * xr.ones_like(obs_2020.longitude)
natural_area = float(area.where(natural_2020).sum())
print(f"Natural land (HM <= 0.1) in the window in 2020: {natural_area:,.0f} km²")

Now we loop over the four forecast years and compute (a) the expected area of natural land lost and (b) the area of natural land whose 97.5th percentile exceeds 0.1, our "plausible risk" screen.

In [ ]:
rows = []
for year in fc.year.values:
    q = fc.hm.sel(year=year, **window).load()
    p = prob_exceed(q, 0.1).where(natural_2020)
    upper = q.sel(percentile=97.5).where(natural_2020)
    rows.append(dict(
        year=int(year),
        expected_loss_km2=float((p * area).sum()),
        upper_screen_km2=float(area.where(upper > 0.1).sum()),
    ))

for r in rows:
    print(f"{r['year']}:  expected loss {r['expected_loss_km2']:8,.0f} km² ({100 * r['expected_loss_km2'] / natural_area:4.1f}%)   "
          f"| natural land with 97.5th pct > 0.1: {r['upper_screen_km2']:8,.0f} km² ({100 * r['upper_screen_km2'] / natural_area:4.1f}%)")

In [ ]:
yrs = [r["year"] for r in rows]
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=110)
ax.bar([y - 0.9 for y in yrs], [100 * r["expected_loss_km2"] / natural_area for r in rows], width=1.8,
       color="#641a80", label="Expected loss (sum of pixel area × probability)")
ax.bar([y + 0.9 for y in yrs], [100 * r["upper_screen_km2"] / natural_area for r in rows], width=1.8,
       color="#fe9f6d", label="Natural land where the 97.5th percentile exceeds 0.1")
ax.set(xticks=yrs, xlabel="Forecast year", ylabel="% of 2020 natural land",
       title="Natural land at risk in central Kenya")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

The purple bars are the best estimate of how much natural land will be lost, and the orange bars show how far the plausible risk extends. The gap between them is the point of a probabilistic forecast. The orange bars are **not** a worst-case total: they mark where substantial modification is plausible and precautionary attention may be warranted, not an amount of land expected to be lost together.

**Scaling up.** The same code works for any region: swap `window` for your area of interest, or loop over ecoregions or countries. For continental or global extents, open the groups with `chunks={}` to get dask arrays and run `prob_exceed` with `dask="parallelized"` in `xr.apply_ufunc`. Each 256 × 320 pixel chunk is independent, so the work parallelises well across a cluster in `us-east-1`. Report areas after accounting for pixel area (as above) or after reprojecting to an equal-area grid. The paper uses EPSG:6933.

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**What is the *joint* risk to a whole landscape? Can we turn per-pixel forecast distributions into spatially coherent scenarios?**

Our forecasts give every pixel a well-calibrated distribution: in held-out tests, 95% prediction intervals contained 92–96% of observations. But they say nothing about how pixels move *together*. Many conservation questions are joint questions:

* What is the 90th-percentile area of natural land an ecoregion could lose by 2040?
* What is the chance that a wildlife corridor is cut somewhere along its length?
* What is the probability that a country's natural-land target becomes unattainable?

Answering them needs **realisations**: many equally plausible maps of HM in 2040 whose pixel values follow the stored marginal distributions *and* show realistic spatial structure. Change in HM is clustered. It spreads out from roads, settlements and agricultural frontiers rather than appearing in scattered single pixels.

**Advice for getting started**

1. **Sampling is easy, dependence is the hard part.** Given a uniform random number *u* for a pixel, interpolating its quantile function at *u* gives a valid draw from its forecast distribution. Drawing *u* independently per pixel gives salt-and-pepper maps with far too little regional variance. The task is to generate spatially correlated *u* fields.
2. **Use a copula.** A Gaussian copula is a natural first step: simulate a correlated Gaussian random field, convert it to uniforms with the normal CDF, then push the uniforms through each pixel's quantile function. The key unknown is the correlation structure: its length scale, anisotropy, and whether it should depend on HM level or distance to past change.
3. **Learn the dependence from history.** The `hm_observed` group gives seven epochs (1990–2020). The spatial autocorrelation of observed 5-, 10- and 20-year changes is a direct, data-driven target for the copula's correlation function. An alternative is the *Schaake shuffle*: reorder samples so that their spatial rank structure copies historical change fields.
4. **Validate against regional totals.** A good scenario generator should reproduce the *distribution of regional change*. Check, for example, whether observed 2000–2020 natural-land loss per ecoregion falls inside your ensemble's regional intervals about as often as it should.
5. **Mind the scale.** The globe has ~185 million land pixels. Work tile by tile, or by ecoregion, and use dask on EC2 in `us-east-1` next to the data. A short-range correlation length (tens of kilometres) keeps random-field simulation tractable with FFT-based methods.

Spatially coherent ensembles would let planners attach honest uncertainty to the area-based targets of the Kunming-Montreal Global Biodiversity Framework. Open an issue on [our GitHub repository](https://github.com/GMoncrieff/open-data-examples/issues) if you take this on. We would love to hear about it.